This notebook generates the well, FOV, timepoint image-sets that still need to be run. 
Generating this loadfile of sorts cuts down on computation time by not running the same image-sets multiple times.
This is a super archaeic form of preemptive caching, but it works.

In [1]:
import argparse
import os
import pathlib
import sys
import time

import natsort
import pandas as pd
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm
image_based_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_based_dir = image_based_dir / "processed_data"
print(image_based_dir)

/home/lippincm/mnt/bandicoot/live_cell_timelapse_pyroptosis_project_data/processed_data


In [ ]:
# get a list of the well_fov directories for each patient
well_fov_dirs = [
    x
    for x in pathlib.Path(f"{image_based_dir}/1.illumination_corrected_files").iterdir()
    if x.is_dir()
]
print(well_fov_dirs)
# get timepoints per well_fov
well_fov_timepoint_raw_file_paths = []
for well_fov_dir in tqdm.tqdm(well_fov_dirs):
    timepoint_dirs = sorted(well_fov_dir.glob(f"*/"))
    for timepoint_dir in timepoint_dirs:
        well_fov_timepoint_raw_file_paths.append(timepoint_dir)
print(well_fov_timepoint_raw_file_paths)
well_fov_df = pd.DataFrame(
    {
        "well_fov_timepoint": [x for x in well_fov_timepoint_raw_file_paths],
    }
)
well_fov_df["well_fov"] = well_fov_df["well_fov_timepoint"].apply(
    lambda x: "_".join(x.name.split("_")[:2])
)
well_fov_df["timepoint"] = well_fov_df["well_fov_timepoint"].apply(
    lambda x: x.name.split("_")[2]
)
# keep T prefix and pad only the numeric part to 4 digits (e.g., T9 -> T0009)
well_fov_df["timepoint"] = well_fov_df["timepoint"].apply(
    lambda x: f"T{str(x)[1:].zfill(4)}"
    if str(x).startswith("T")
    else f"T{str(x).zfill(4)}"
)
well_fov_df.drop_duplicates(subset=["well_fov", "timepoint"], inplace=True)

  0%|          | 0/224 [00:00<?, ?it/s]

In [ ]:
print(well_fov_df)

In [3]:
# Build expected output sqlite path per well_fov + timepoint row
well_fov_df["output_path"] = well_fov_df["well_fov_timepoint"].apply(
    lambda x: str(pathlib.Path(x).parents[2] / "3.extracted_features")
)

well_fov_df["output_file_path"] = [
    str(
        pathlib.Path(output_path)
        / f"{well_fov}_{timepoint}"
        / f"{well_fov}_{timepoint}.sqlite"
    )
    for output_path, well_fov, timepoint in zip(
        well_fov_df["output_path"],
        well_fov_df["well_fov"],
        well_fov_df["timepoint"],
    )
]

well_fov_df["output_file_path_exists"] = well_fov_df["output_file_path"].apply(
    lambda x: pathlib.Path(x).exists()
)

In [4]:
# sort the df
well_fov_df.sort_values(by=["well_fov", "timepoint"], inplace=True)
# natural sort the df by well_fov and timepoint
well_fov_df = well_fov_df.iloc[
    natsort.index_natsorted(
        well_fov_df["well_fov"].astype(str) + "_" + well_fov_df["timepoint"].astype(str)
    )
].reset_index(drop=True)
well_fov_df.head()

,well_fov_timepoint,well_fov,timepoint,output_path,output_file_path,output_file_path_exists
0,/home/lippincm/mnt/bandicoot/live_cell_timelap...,B2_1,T0001,/home/lippincm/mnt/bandicoot/live_cell_timelap...,/home/lippincm/mnt/bandicoot/live_cell_timelap...,True
1,/home/lippincm/mnt/bandicoot/live_cell_timelap...,B2_1,T0002,/home/lippincm/mnt/bandicoot/live_cell_timelap...,/home/lippincm/mnt/bandicoot/live_cell_timelap...,True
2,/home/lippincm/mnt/bandicoot/live_cell_timelap...,B2_1,T0003,/home/lippincm/mnt/bandicoot/live_cell_timelap...,/home/lippincm/mnt/bandicoot/live_cell_timelap...,True
3,/home/lippincm/mnt/bandicoot/live_cell_timelap...,B2_1,T0004,/home/lippincm/mnt/bandicoot/live_cell_timelap...,/home/lippincm/mnt/bandicoot/live_cell_timelap...,True
4,/home/lippincm/mnt/bandicoot/live_cell_timelap...,B2_1,T0005,/home/lippincm/mnt/bandicoot/live_cell_timelap...,/home/lippincm/mnt/bandicoot/live_cell_timelap...,True


In [5]:
# find the number of well fov timepoints still needed
to_run = well_fov_df.loc[well_fov_df["output_file_path_exists"] == False]
completed = well_fov_df.loc[well_fov_df["output_file_path_exists"] == True]
print(f"Number of well fov timepoints still needed: {len(to_run)}")
print(f"Number of well fov timepoints completed: {len(completed)}")
print(f"Progress: {len(completed) / len(well_fov_df) * 100:.2f}%")

Number of well fov timepoints still needed: 22838
Number of well fov timepoints completed: 10
Progress: 0.04%


In [6]:
well_fov_df["well_fov"].unique()

array(['B2_1', 'B2_2', 'B2_3', 'B2_4', 'B3_1', 'B3_2', 'B3_3', 'B3_4',
       'B4_1', 'B4_2', 'B4_3', 'B4_4', 'B5_1', 'B5_2', 'B5_3', 'B5_4',
       'C2_1', 'C2_2', 'C2_3', 'C2_4', 'C3_1', 'C3_2', 'C3_3', 'C3_4',
       'C4_1', 'C4_2', 'C4_3', 'C4_4', 'C5_1', 'C5_2', 'C5_3', 'C5_4',
       'D2_1', 'D2_2', 'D2_3', 'D2_4', 'D3_1', 'D3_2', 'D3_3', 'D3_4',
       'D4_1', 'D4_2', 'D4_3', 'D4_4', 'D5_1', 'D5_2', 'D5_3', 'D5_4',
       'E2_1', 'E2_2', 'E2_3', 'E2_4', 'E3_1', 'E3_2', 'E3_3', 'E3_4',
       'E4_1', 'E4_2', 'E4_3', 'E4_4', 'E5_1', 'E5_2', 'E5_3', 'E5_4',
       'F2_1', 'F2_2', 'F2_3', 'F2_4', 'F3_1', 'F3_2', 'F3_3', 'F3_4',
       'F4_1', 'F4_2', 'F4_3', 'F4_4', 'F5_1', 'F5_2', 'F5_3', 'F5_4',
       'G2_1', 'G2_2', 'G2_3', 'G2_4', 'G3_1', 'G3_2', 'G3_3', 'G3_4',
       'G4_1', 'G4_2', 'G4_3', 'G4_4', 'G5_1', 'G5_2', 'G5_3', 'G5_4',
       'H2_1', 'H2_2', 'H2_3', 'H2_4', 'H3_1', 'H3_2', 'H3_3', 'H3_4',
       'H4_1', 'H4_2', 'H4_3', 'H4_4', 'H5_1', 'H5_2', 'H5_3', 'H5_4',
      